In [ ]:
# 06. Fire Risk Engine

## Objective
Develop a modular risk-scoring engine for FireWatch AI using existing fire and smoke detection results and temporal analysis.

## Key Responsibilities
- Combine fire and smoke detection signals.
- Consider detection confidence and temporal persistence.
- Assign configurable risk scores and risk levels.
- Generate structured, auditable risk reports.
- Validate outputs and handle missing or invalid inputs.


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
from pathlib import Path
import pandas as pd

# Project paths
PROJECT = Path("/content/drive/MyDrive/FireWatch-AI")

MODEL_PATH = PROJECT / "runs/fire_smoke/yolo26n_fire_smoke_v1/weights/best.pt"
TEMPORAL_PATH = PROJECT / "outputs/temporal_analysis/temporal_final_report.csv"

OUTPUT_DIR = PROJECT / "outputs/fire_risk_engine"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Validate required inputs
assert MODEL_PATH.is_file(), f"Trained model missing: {MODEL_PATH}"
assert TEMPORAL_PATH.is_file(), f"Temporal report missing: {TEMPORAL_PATH}"

temporal_df = pd.read_csv(TEMPORAL_PATH)

required_columns = {
    "video_name", "class", "detected_frames",
    "sampled_frames", "avg_confidence", "detection_rate"
}

missing_columns = required_columns - set(temporal_df.columns)
assert not missing_columns, f"Missing columns: {missing_columns}"
assert not temporal_df.empty, "Temporal report is empty"

print("Project setup: PASS")
print("Model exists:", MODEL_PATH.name)
print("Videos:", temporal_df["video_name"].nunique())
print("Temporal records:", len(temporal_df))
print("Output directory ready:", OUTPUT_DIR)

Project setup: PASS
Model exists: best.pt
Videos: 2
Temporal records: 4
Output directory ready: /content/drive/MyDrive/FireWatch-AI/outputs/fire_risk_engine


In [ ]:
#Build Fire Risk Engine

In [3]:
import numpy as np
import pandas as pd

# Configurable weights
WEIGHTS = {
    "fire": 0.70,
    "smoke": 0.30
}

RISK_THRESHOLDS = {
    "Low": (0, 24),
    "Medium": (25, 49),
    "High": (50, 74),
    "Critical": (75, 100)
}

def calculate_risk(video_df):
    signals = {}

    for cls in WEIGHTS:
        row = video_df[video_df["class"].str.lower() == cls]

        if row.empty:
            signals[cls] = 0.0
            continue

        row = row.iloc[0]
        rate = float(np.clip(row["detection_rate"], 0, 1))
        confidence = float(np.clip(row["avg_confidence"], 0, 1))

        # Confidence-weighted detection signal
        signals[cls] = rate * confidence

    # Weighted score from 0 to 100
    score = round(
        100 * sum(WEIGHTS[cls] * signals[cls] for cls in WEIGHTS),
        2
    )

    # Risk category
    level = next(
        name for name, (low, high) in RISK_THRESHOLDS.items()
        if low <= score <= high
    )

    return score, level

results = []

for video_name, group in temporal_df.groupby("video_name"):
    score, level = calculate_risk(group)

    results.append({
        "video_name": video_name,
        "risk_score": score,
        "risk_level": level,
        "fire_detection_rate": float(
            group.loc[group["class"].str.lower() == "fire", "detection_rate"].max()
        ) if (group["class"].str.lower() == "fire").any() else 0.0,
        "smoke_detection_rate": float(
            group.loc[group["class"].str.lower() == "smoke", "detection_rate"].max()
        ) if (group["class"].str.lower() == "smoke").any() else 0.0
    })

risk_df = pd.DataFrame(results)
risk_df.to_csv(OUTPUT_DIR / "video_risk_report.csv", index=False)

display(risk_df)
print("Saved:", OUTPUT_DIR / "video_risk_report.csv")

,video_name,risk_score,risk_level,fire_detection_rate,smoke_detection_rate
0,posVideo1.868.avi,37.38,Medium,1.0,0.000
1,testpos01.817.avi,8.98,Low,0.0,0.634


Saved: /content/drive/MyDrive/FireWatch-AI/outputs/fire_risk_engine/video_risk_report.csv


In [4]:
import pandas as pd

risk_path = OUTPUT_DIR / "video_risk_report.csv"
risk_df = pd.read_csv(risk_path)

def assign_safety_level(row):
    if row["fire_detection_rate"] > 0:
        return "High"
    if row["smoke_detection_rate"] > 0:
        return "Medium" if row["risk_score"] >= 10 else "Low"
    return "Low"

risk_df["risk_level"] = risk_df.apply(assign_safety_level, axis=1)

# Basic validation
assert risk_df["risk_score"].between(0, 100).all()
assert risk_df["risk_level"].isin(["Low", "Medium", "High", "Critical"]).all()

risk_df.to_csv(risk_path, index=False)

print("Risk rules validation: PASS")
display(risk_df)
print("Updated:", risk_path)

Risk rules validation: PASS


,video_name,risk_score,risk_level,fire_detection_rate,smoke_detection_rate
0,posVideo1.868.avi,37.38,High,1.0,0.000
1,testpos01.817.avi,8.98,Low,0.0,0.634


Updated: /content/drive/MyDrive/FireWatch-AI/outputs/fire_risk_engine/video_risk_report.csv


In [5]:
from pathlib import Path
import pandas as pd

risk_path = OUTPUT_DIR / "video_risk_report.csv"
risk_df = pd.read_csv(risk_path)

# Automated checks
assert not risk_df.empty
assert risk_df["video_name"].notna().all()
assert risk_df["risk_score"].between(0, 100).all()
assert risk_df["risk_level"].isin(
    ["Low", "Medium", "High", "Critical"]
).all()
assert risk_df["video_name"].is_unique

# Safety rule: detected fire must never be labelled Low or Medium
fire_rows = risk_df[risk_df["fire_detection_rate"] > 0]
assert fire_rows["risk_level"].isin(["High", "Critical"]).all()

summary_path = OUTPUT_DIR / "fire_risk_engine_validation.txt"
summary_path.write_text(
    "FireWatch AI - Fire Risk Engine Validation\n"
    "==========================================\n"
    "Risk score range: PASS\n"
    "Risk level values: PASS\n"
    "Unique video records: PASS\n"
    "Fire safety rule: PASS\n"
    f"Videos processed: {len(risk_df)}\n"
    "Overall status: PASS\n"
    "\nNote: Prototype risk estimates only; not certified fire-safety decisions.\n"
)

print("ALL TESTS PASSED")
print("Videos:", len(risk_df))
print("Risk report:", risk_path)
print("Validation report:", summary_path)

ALL TESTS PASSED
Videos: 2
Risk report: /content/drive/MyDrive/FireWatch-AI/outputs/fire_risk_engine/video_risk_report.csv
Validation report: /content/drive/MyDrive/FireWatch-AI/outputs/fire_risk_engine/fire_risk_engine_validation.txt


In [6]:
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/FireWatch-AI")
VIDEO_ROOT = PROJECT / "datasets/videos/firesense"

for category in ["fire", "smoke"]:
    folder = VIDEO_ROOT / category
    videos = [
        p for p in folder.rglob("*")
        if p.is_file()
        and p.suffix.lower() in {".avi", ".mp4", ".mov", ".mkv"}
    ]
    print(f"{category}: {len(videos)} videos")
    print([p.name for p in videos[:10]])

fire: 27 videos
['negsVideo1.858.avi', 'negsVideo2.859.avi', 'negsVideo3.860.avi', 'negsVideo4.861.avi', 'negsVideo5.862.avi', 'negsVideo6.863.avi', 'negsVideo7.864.avi', 'negsVideo8.865.avi', 'negsVideo9.866.avi', 'negsVideo10.1072.avi']
smoke: 22 videos
['testneg01.807.avi', 'testneg02.808.avi', 'testneg03.809.avi', 'testneg04.810.avi', 'testneg05.811.avi', 'testneg06.812.avi', 'testneg07.813.avi', 'testneg08.814.avi', 'testneg09.815.avi', 'testpos01.817.avi']


In [8]:
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/FireWatch-AI")
VIDEO_ROOT = PROJECT / "datasets/videos/firesense"

all_videos = [
    p for p in VIDEO_ROOT.rglob("*")
    if p.is_file()
    and p.suffix.lower() in {".avi", ".mp4", ".mov", ".mkv"}
]

print("Total videos found:", len(all_videos))

for p in all_videos:
    if "pos" in p.name.lower() or "neg" in p.name.lower():
        print(p.name, "->", p.parent)

Total videos found: 49
negsVideo1.858.avi -> /content/drive/MyDrive/FireWatch-AI/datasets/videos/firesense/fire/neg
negsVideo2.859.avi -> /content/drive/MyDrive/FireWatch-AI/datasets/videos/firesense/fire/neg
negsVideo3.860.avi -> /content/drive/MyDrive/FireWatch-AI/datasets/videos/firesense/fire/neg
negsVideo4.861.avi -> /content/drive/MyDrive/FireWatch-AI/datasets/videos/firesense/fire/neg
negsVideo5.862.avi -> /content/drive/MyDrive/FireWatch-AI/datasets/videos/firesense/fire/neg
negsVideo6.863.avi -> /content/drive/MyDrive/FireWatch-AI/datasets/videos/firesense/fire/neg
negsVideo7.864.avi -> /content/drive/MyDrive/FireWatch-AI/datasets/videos/firesense/fire/neg
negsVideo8.865.avi -> /content/drive/MyDrive/FireWatch-AI/datasets/videos/firesense/fire/neg
negsVideo9.866.avi -> /content/drive/MyDrive/FireWatch-AI/datasets/videos/firesense/fire/neg
negsVideo10.1072.avi -> /content/drive/MyDrive/FireWatch-AI/datasets/videos/firesense/fire/neg
negsVideo11.1073.avi -> /content/drive/MyDriv

In [9]:
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/FireWatch-AI")
VIDEO_ROOT = PROJECT / "datasets/videos/firesense"

selection = [
    ("fire", "pos", "posVideo1.868.avi", "fire_positive"),
    ("fire", "pos", "posVideo2.871.avi", "fire_positive"),
    ("fire", "neg", "negsVideo1.858.avi", "negative"),
    ("smoke", "pos", "testpos01.817.avi", "smoke_positive"),
    ("smoke", "pos", "testpos02.818.avi", "smoke_positive"),
    ("smoke", "neg", "testneg01.807.avi", "negative"),
    ("smoke", "neg", "testneg02.808.avi", "negative"),
]

test_videos = []

for category, split, filename, label in selection:
    path = VIDEO_ROOT / category / split / filename
    assert path.is_file(), f"Missing video: {path}"
    test_videos.append({
        "video_path": str(path),
        "video_name": filename,
        "video_category": category,
        "ground_truth": label,
    })

print("Selected videos:", len(test_videos))
for item in test_videos:
    print(item["ground_truth"], "|", item["video_name"])

Selected videos: 7
fire_positive | posVideo1.868.avi
fire_positive | posVideo2.871.avi
negative | negsVideo1.858.avi
smoke_positive | testpos01.817.avi
smoke_positive | testpos02.818.avi
negative | testneg01.807.avi
negative | testneg02.808.avi


In [11]:
%pip install -q ultralytics

In [12]:
import ultralytics
print("Ultralytics version:", ultralytics.__version__)

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Ultralytics version: 8.4.190


In [13]:
from pathlib import Path
import pandas as pd
import cv2
from ultralytics import YOLO

PROJECT = Path("/content/drive/MyDrive/FireWatch-AI")
VIDEO_ROOT = PROJECT / "datasets/videos/firesense"
MODEL_PATH = PROJECT / "runs/fire_smoke/yolo26n_fire_smoke_v1/weights/best.pt"
OUTPUT_DIR = PROJECT / "outputs/fire_risk_engine"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RESULT_PATH = OUTPUT_DIR / "seven_video_evaluation.csv"

# First video only
video_path = VIDEO_ROOT / "fire/pos/posVideo1.868.avi"
assert MODEL_PATH.is_file(), f"Model missing: {MODEL_PATH}"
assert video_path.is_file(), f"Video missing: {video_path}"

model = YOLO(str(MODEL_PATH))
cap = cv2.VideoCapture(str(video_path))
assert cap.isOpened(), f"Cannot open: {video_path.name}"

sampled = 0
counts = {"fire": 0, "smoke": 0}
confidence_sum = {"fire": 0.0, "smoke": 0.0}
frame_index = 0

try:
    while True:
        ok, frame = cap.read()
        if not ok:
            break

        if frame_index % 15 == 0:
            result = model.predict(
                frame, conf=0.25, imgsz=416, verbose=False
            )[0]
            sampled += 1
            seen = set()

            for box in result.boxes:
                cls = model.names[int(box.cls[0])].lower()
                if cls in counts and cls not in seen:
                    counts[cls] += 1
                    confidence_sum[cls] += float(box.conf[0])
                    seen.add(cls)

        frame_index += 1
finally:
    cap.release()

rows = []
for cls in counts:
    rows.append({
        "video_name": video_path.name,
        "video_category": "fire",
        "ground_truth": "fire_positive",
        "detected_class": cls,
        "sampled_frames": sampled,
        "detected_frames": counts[cls],
        "detection_rate": round(counts[cls] / sampled, 4) if sampled else 0,
        "avg_confidence": round(
            confidence_sum[cls] / counts[cls], 4
        ) if counts[cls] else 0
    })

pd.DataFrame(rows).to_csv(RESULT_PATH, index=False)

print("Completed:", video_path.name)
print("Sampled frames:", sampled)
display(pd.DataFrame(rows))
print("Saved:", RESULT_PATH)

Completed: posVideo1.868.avi
Sampled frames: 20


,video_name,video_category,ground_truth,detected_class,sampled_frames,detected_frames,detection_rate,avg_confidence
0,posVideo1.868.avi,fire,fire_positive,fire,20,20,1.0,0.5668
1,posVideo1.868.avi,fire,fire_positive,smoke,20,0,0.0,0.0000


Saved: /content/drive/MyDrive/FireWatch-AI/outputs/fire_risk_engine/seven_video_evaluation.csv


In [14]:
from pathlib import Path
import pandas as pd
import cv2
from ultralytics import YOLO

PROJECT = Path("/content/drive/MyDrive/FireWatch-AI")
VIDEO_ROOT = PROJECT / "datasets/videos/firesense"
MODEL_PATH = PROJECT / "runs/fire_smoke/yolo26n_fire_smoke_v1/weights/best.pt"
RESULT_PATH = PROJECT / "outputs/fire_risk_engine/seven_video_evaluation.csv"

video_path = VIDEO_ROOT / "fire/pos/posVideo2.871.avi"
assert video_path.is_file(), f"Video missing: {video_path}"

model = YOLO(str(MODEL_PATH))
cap = cv2.VideoCapture(str(video_path))
assert cap.isOpened(), f"Cannot open: {video_path.name}"

counts = {"fire": 0, "smoke": 0}
confidence_sum = {"fire": 0.0, "smoke": 0.0}
sampled = frame_index = 0

try:
    while True:
        ok, frame = cap.read()
        if not ok:
            break

        if frame_index % 15 == 0:
            result = model.predict(frame, conf=0.25, imgsz=416, verbose=False)[0]
            sampled += 1
            seen = set()

            for box in result.boxes:
                cls = model.names[int(box.cls[0])].lower()
                if cls in counts and cls not in seen:
                    counts[cls] += 1
                    confidence_sum[cls] += float(box.conf[0])
                    seen.add(cls)

        frame_index += 1
finally:
    cap.release()

rows = [{
    "video_name": video_path.name,
    "video_category": "fire",
    "ground_truth": "fire_positive",
    "detected_class": cls,
    "sampled_frames": sampled,
    "detected_frames": counts[cls],
    "detection_rate": round(counts[cls] / sampled, 4) if sampled else 0,
    "avg_confidence": round(confidence_sum[cls] / counts[cls], 4) if counts[cls] else 0
} for cls in counts]

new_rows = pd.DataFrame(rows)
old_rows = pd.read_csv(RESULT_PATH) if RESULT_PATH.exists() else pd.DataFrame()
pd.concat([old_rows, new_rows], ignore_index=True).to_csv(RESULT_PATH, index=False)

print("Completed:", video_path.name, "| Sampled frames:", sampled)
display(new_rows)
print("Unique videos saved:", pd.read_csv(RESULT_PATH)["video_name"].nunique())

Completed: posVideo2.871.avi | Sampled frames: 34


,video_name,video_category,ground_truth,detected_class,sampled_frames,detected_frames,detection_rate,avg_confidence
0,posVideo2.871.avi,fire,fire_positive,fire,34,34,1.0000,0.7974
1,posVideo2.871.avi,fire,fire_positive,smoke,34,1,0.0294,0.2501


Unique videos saved: 2


In [15]:
from pathlib import Path
import pandas as pd
import cv2
from ultralytics import YOLO

PROJECT = Path("/content/drive/MyDrive/FireWatch-AI")
VIDEO_ROOT = PROJECT / "datasets/videos/firesense"
MODEL_PATH = PROJECT / "runs/fire_smoke/yolo26n_fire_smoke_v1/weights/best.pt"
RESULT_PATH = PROJECT / "outputs/fire_risk_engine/seven_video_evaluation.csv"

video_path = VIDEO_ROOT / "fire/neg/negsVideo1.858.avi"
assert video_path.is_file(), f"Video missing: {video_path}"

existing = pd.read_csv(RESULT_PATH)
if video_path.name in existing["video_name"].values:
    print("Already evaluated:", video_path.name)
else:
    model = YOLO(str(MODEL_PATH))
    cap = cv2.VideoCapture(str(video_path))
    assert cap.isOpened(), f"Cannot open: {video_path.name}"

    counts = {"fire": 0, "smoke": 0}
    confidence_sum = {"fire": 0.0, "smoke": 0.0}
    sampled = frame_index = 0

    try:
        while True:
            ok, frame = cap.read()
            if not ok:
                break

            if frame_index % 15 == 0:
                result = model.predict(
                    frame, conf=0.25, imgsz=416, verbose=False
                )[0]
                sampled += 1
                seen = set()

                for box in result.boxes:
                    cls = model.names[int(box.cls[0])].lower()
                    if cls in counts and cls not in seen:
                        counts[cls] += 1
                        confidence_sum[cls] += float(box.conf[0])
                        seen.add(cls)

            frame_index += 1
    finally:
        cap.release()

    rows = [{
        "video_name": video_path.name,
        "video_category": "fire",
        "ground_truth": "negative",
        "detected_class": cls,
        "sampled_frames": sampled,
        "detected_frames": counts[cls],
        "detection_rate": round(counts[cls] / sampled, 4) if sampled else 0,
        "avg_confidence": round(
            confidence_sum[cls] / counts[cls], 4
        ) if counts[cls] else 0
    } for cls in counts]

    pd.concat(
        [existing, pd.DataFrame(rows)], ignore_index=True
    ).to_csv(RESULT_PATH, index=False)

    print("Completed:", video_path.name, "| Sampled frames:", sampled)
    display(pd.DataFrame(rows))
    print("Unique videos saved:", pd.read_csv(RESULT_PATH)["video_name"].nunique())

Completed: negsVideo1.858.avi | Sampled frames: 303


,video_name,video_category,ground_truth,detected_class,sampled_frames,detected_frames,detection_rate,avg_confidence
0,negsVideo1.858.avi,fire,negative,fire,303,15,0.0495,0.4329
1,negsVideo1.858.avi,fire,negative,smoke,303,2,0.0066,0.2775


Unique videos saved: 3


In [16]:
from pathlib import Path
import pandas as pd
import cv2
from ultralytics import YOLO

PROJECT = Path("/content/drive/MyDrive/FireWatch-AI")
VIDEO_ROOT = PROJECT / "datasets/videos/firesense"
MODEL_PATH = PROJECT / "runs/fire_smoke/yolo26n_fire_smoke_v1/weights/best.pt"
RESULT_PATH = PROJECT / "outputs/fire_risk_engine/seven_video_evaluation.csv"

video_path = VIDEO_ROOT / "smoke/pos/testpos01.817.avi"
existing = pd.read_csv(RESULT_PATH)

if video_path.name in existing["video_name"].values:
    print("Already evaluated:", video_path.name)
else:
    model = YOLO(str(MODEL_PATH))
    cap = cv2.VideoCapture(str(video_path))
    assert cap.isOpened(), f"Cannot open: {video_path.name}"

    counts = {"fire": 0, "smoke": 0}
    confidence_sum = {"fire": 0.0, "smoke": 0.0}
    sampled = frame_index = 0

    try:
        while True:
            ok, frame = cap.read()
            if not ok:
                break

            if frame_index % 15 == 0:
                result = model.predict(
                    frame, conf=0.25, imgsz=416, verbose=False
                )[0]
                sampled += 1
                seen = set()

                for box in result.boxes:
                    cls = model.names[int(box.cls[0])].lower()
                    if cls in counts and cls not in seen:
                        counts[cls] += 1
                        confidence_sum[cls] += float(box.conf[0])
                        seen.add(cls)

            frame_index += 1
    finally:
        cap.release()

    rows = [{
        "video_name": video_path.name,
        "video_category": "smoke",
        "ground_truth": "smoke_positive",
        "detected_class": cls,
        "sampled_frames": sampled,
        "detected_frames": counts[cls],
        "detection_rate": round(counts[cls] / sampled, 4) if sampled else 0,
        "avg_confidence": round(
            confidence_sum[cls] / counts[cls], 4
        ) if counts[cls] else 0
    } for cls in counts]

    pd.concat([existing, pd.DataFrame(rows)], ignore_index=True).to_csv(
        RESULT_PATH, index=False
    )

    print("Completed:", video_path.name, "| Sampled frames:", sampled)
    display(pd.DataFrame(rows))
    print("Unique videos saved:", pd.read_csv(RESULT_PATH)["video_name"].nunique())

Completed: testpos01.817.avi | Sampled frames: 28


,video_name,video_category,ground_truth,detected_class,sampled_frames,detected_frames,detection_rate,avg_confidence
0,testpos01.817.avi,smoke,smoke_positive,fire,28,0,0.0000,0.000
1,testpos01.817.avi,smoke,smoke_positive,smoke,28,16,0.5714,0.713


Unique videos saved: 4


In [17]:
from pathlib import Path
import pandas as pd
import cv2
from ultralytics import YOLO

PROJECT = Path("/content/drive/MyDrive/FireWatch-AI")
VIDEO_ROOT = PROJECT / "datasets/videos/firesense"
MODEL_PATH = PROJECT / "runs/fire_smoke/yolo26n_fire_smoke_v1/weights/best.pt"
RESULT_PATH = PROJECT / "outputs/fire_risk_engine/seven_video_evaluation.csv"

video_path = VIDEO_ROOT / "smoke/pos/testpos02.818.avi"
existing = pd.read_csv(RESULT_PATH)

if video_path.name in existing["video_name"].values:
    print("Already evaluated:", video_path.name)
else:
    model = YOLO(str(MODEL_PATH))
    cap = cv2.VideoCapture(str(video_path))
    assert cap.isOpened(), f"Cannot open: {video_path.name}"

    counts = {"fire": 0, "smoke": 0}
    conf_sum = {"fire": 0.0, "smoke": 0.0}
    sampled = frame_index = 0

    try:
        while True:
            ok, frame = cap.read()
            if not ok:
                break
            if frame_index % 15 == 0:
                result = model.predict(
                    frame, conf=0.25, imgsz=416, verbose=False
                )[0]
                sampled += 1
                seen = set()
                for box in result.boxes:
                    cls = model.names[int(box.cls[0])].lower()
                    if cls in counts and cls not in seen:
                        counts[cls] += 1
                        conf_sum[cls] += float(box.conf[0])
                        seen.add(cls)
            frame_index += 1
    finally:
        cap.release()

    rows = [{
        "video_name": video_path.name,
        "video_category": "smoke",
        "ground_truth": "smoke_positive",
        "detected_class": cls,
        "sampled_frames": sampled,
        "detected_frames": counts[cls],
        "detection_rate": round(counts[cls] / sampled, 4) if sampled else 0,
        "avg_confidence": round(conf_sum[cls] / counts[cls], 4) if counts[cls] else 0
    } for cls in counts]

    pd.concat([existing, pd.DataFrame(rows)], ignore_index=True).to_csv(
        RESULT_PATH, index=False
    )
    print("Completed:", video_path.name, "| Sampled frames:", sampled)
    display(pd.DataFrame(rows))
    print("Unique videos saved:", pd.read_csv(RESULT_PATH)["video_name"].nunique())




Completed: testpos02.818.avi | Sampled frames: 116


,video_name,video_category,ground_truth,detected_class,sampled_frames,detected_frames,detection_rate,avg_confidence
0,testpos02.818.avi,smoke,smoke_positive,fire,116,43,0.3707,0.6743
1,testpos02.818.avi,smoke,smoke_positive,smoke,116,0,0.0000,0.0000


Unique videos saved: 5


In [18]:
from pathlib import Path
import pandas as pd
import cv2
from ultralytics import YOLO

PROJECT = Path("/content/drive/MyDrive/FireWatch-AI")
VIDEO_ROOT = PROJECT / "datasets/videos/firesense"
MODEL_PATH = PROJECT / "runs/fire_smoke/yolo26n_fire_smoke_v1/weights/best.pt"
RESULT_PATH = PROJECT / "outputs/fire_risk_engine/seven_video_evaluation.csv"

video_path = VIDEO_ROOT / "smoke/neg/testneg01.807.avi"
existing = pd.read_csv(RESULT_PATH)

if video_path.name in existing["video_name"].values:
    print("Already evaluated:", video_path.name)
else:
    model = YOLO(str(MODEL_PATH))
    cap = cv2.VideoCapture(str(video_path))
    assert cap.isOpened(), f"Cannot open: {video_path.name}"

    counts = {"fire": 0, "smoke": 0}
    conf_sum = {"fire": 0.0, "smoke": 0.0}
    sampled = frame_index = 0

    try:
        while True:
            ok, frame = cap.read()
            if not ok:
                break
            if frame_index % 15 == 0:
                result = model.predict(
                    frame, conf=0.25, imgsz=416, verbose=False
                )[0]
                sampled += 1
                seen = set()
                for box in result.boxes:
                    cls = model.names[int(box.cls[0])].lower()
                    if cls in counts and cls not in seen:
                        counts[cls] += 1
                        conf_sum[cls] += float(box.conf[0])
                        seen.add(cls)
            frame_index += 1
    finally:
        cap.release()

    rows = [{
        "video_name": video_path.name,
        "video_category": "smoke",
        "ground_truth": "negative",
        "detected_class": cls,
        "sampled_frames": sampled,
        "detected_frames": counts[cls],
        "detection_rate": round(counts[cls] / sampled, 4) if sampled else 0,
        "avg_confidence": round(conf_sum[cls] / counts[cls], 4) if counts[cls] else 0
    } for cls in counts]

    pd.concat([existing, pd.DataFrame(rows)], ignore_index=True).to_csv(
        RESULT_PATH, index=False
    )
    print("Completed:", video_path.name, "| Sampled frames:", sampled)
    display(pd.DataFrame(rows))
    print("Unique videos saved:", pd.read_csv(RESULT_PATH)["video_name"].nunique())

Completed: testneg01.807.avi | Sampled frames: 17


,video_name,video_category,ground_truth,detected_class,sampled_frames,detected_frames,detection_rate,avg_confidence
0,testneg01.807.avi,smoke,negative,fire,17,0,0.0,0
1,testneg01.807.avi,smoke,negative,smoke,17,0,0.0,0


Unique videos saved: 6


In [19]:
from pathlib import Path
import pandas as pd
import cv2
from ultralytics import YOLO

PROJECT = Path("/content/drive/MyDrive/FireWatch-AI")
VIDEO_ROOT = PROJECT / "datasets/videos/firesense"
MODEL_PATH = PROJECT / "runs/fire_smoke/yolo26n_fire_smoke_v1/weights/best.pt"
OUT = PROJECT / "outputs/fire_risk_engine"
RESULT_PATH = OUT / "seven_video_evaluation.csv"
SUMMARY_PATH = OUT / "seven_video_evaluation_summary.csv"
OUT.mkdir(parents=True, exist_ok=True)

video_path = VIDEO_ROOT / "smoke/neg/testneg02.808.avi"
existing = pd.read_csv(RESULT_PATH) if RESULT_PATH.exists() else pd.DataFrame()

if video_path.name not in existing.get("video_name", pd.Series(dtype=str)).values:
    model = YOLO(str(MODEL_PATH))
    cap = cv2.VideoCapture(str(video_path))
    assert cap.isOpened(), f"Cannot open: {video_path}"

    counts = {"fire": 0, "smoke": 0}
    conf_sum = {"fire": 0.0, "smoke": 0.0}
    sampled = frame_index = 0

    try:
        while True:
            ok, frame = cap.read()
            if not ok:
                break
            if frame_index % 15 == 0:
                result = model.predict(frame, conf=0.25, imgsz=416, verbose=False)[0]
                sampled += 1
                seen = set()
                for box in result.boxes:
                    cls = model.names[int(box.cls[0])].lower()
                    if cls in counts and cls not in seen:
                        counts[cls] += 1
                        conf_sum[cls] += float(box.conf[0])
                        seen.add(cls)
            frame_index += 1
    finally:
        cap.release()

    rows = [{
        "video_name": video_path.name,
        "video_category": "smoke",
        "ground_truth": "negative",
        "detected_class": cls,
        "sampled_frames": sampled,
        "detected_frames": counts[cls],
        "detection_rate": round(counts[cls] / sampled, 4) if sampled else 0,
        "avg_confidence": round(conf_sum[cls] / counts[cls], 4) if counts[cls] else 0
    } for cls in counts]

    existing = pd.concat([existing, pd.DataFrame(rows)], ignore_index=True)
    existing.to_csv(RESULT_PATH, index=False)
    print("Completed final video:", video_path.name)
else:
    print("Final video already evaluated.")

# Consolidated per-video summary
df = pd.read_csv(RESULT_PATH)
summary = df.pivot_table(
    index=["video_name", "video_category", "ground_truth"],
    columns="detected_class",
    values="detection_rate",
    aggfunc="first",
    fill_value=0
).reset_index()
summary.columns.name = None

for cls in ["fire", "smoke"]:
    if cls not in summary.columns:
        summary[cls] = 0.0

summary = summary.rename(columns={
    "fire": "fire_detection_rate",
    "smoke": "smoke_detection_rate"
})
summary.to_csv(SUMMARY_PATH, index=False)

print("\nUnique videos evaluated:", summary["video_name"].nunique())
print("Detailed results:", RESULT_PATH)
print("Video summary:", SUMMARY_PATH)
display(summary)

Completed final video: testneg02.808.avi

Unique videos evaluated: 7
Detailed results: /content/drive/MyDrive/FireWatch-AI/outputs/fire_risk_engine/seven_video_evaluation.csv
Video summary: /content/drive/MyDrive/FireWatch-AI/outputs/fire_risk_engine/seven_video_evaluation_summary.csv


,video_name,video_category,ground_truth,fire_detection_rate,smoke_detection_rate
0,negsVideo1.858.avi,fire,negative,0.0495,0.0066
1,posVideo1.868.avi,fire,fire_positive,1.0000,0.0000
2,posVideo2.871.avi,fire,fire_positive,1.0000,0.0294
3,testneg01.807.avi,smoke,negative,0.0000,0.0000
4,testneg02.808.avi,smoke,negative,0.0000,0.0000
5,testpos01.817.avi,smoke,smoke_positive,0.0000,0.5714
6,testpos02.818.avi,smoke,smoke_positive,0.3707,0.0000


In [20]:
import pandas as pd
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/FireWatch-AI")
OUT = PROJECT / "outputs/fire_risk_engine"

summary_path = OUT / "seven_video_evaluation_summary.csv"
risk_path = OUT / "seven_video_risk_report.csv"

df = pd.read_csv(summary_path)

# Risk score from detection rate and class weights
df["risk_score"] = (
    100 * (
        0.70 * df["fire_detection_rate"]
        + 0.30 * df["smoke_detection_rate"]
    )
).round(2)

# Conservative prototype rule: any fire detection raises risk to High
def risk_level(row):
    if row["fire_detection_rate"] > 0:
        return "Critical" if row["risk_score"] >= 75 else "High"
    if row["smoke_detection_rate"] > 0:
        return "Medium" if row["risk_score"] >= 10 else "Low"
    return "Low"

df["risk_level"] = df.apply(risk_level, axis=1)

# Flag potential class confusion and false alarms for review
def finding(row):
    if row["ground_truth"] == "negative":
        if row["fire_detection_rate"] > 0 or row["smoke_detection_rate"] > 0:
            return "Potential false alarm"
        return "No detections on negative video"
    if row["ground_truth"] == "smoke_positive" and row["fire_detection_rate"] > 0:
        return "Potential fire-smoke confusion"
    if row["ground_truth"] == "smoke_positive" and row["smoke_detection_rate"] == 0:
        return "Smoke missed"
    if row["ground_truth"] == "fire_positive" and row["fire_detection_rate"] == 0:
        return "Fire missed"
    return "Expected class detected"

df["evaluation_finding"] = df.apply(finding, axis=1)

df.to_csv(risk_path, index=False)

print("Videos evaluated:", df["video_name"].nunique())
print("Risk report saved:", risk_path)
display(df[[
    "video_name", "ground_truth", "fire_detection_rate",
    "smoke_detection_rate", "risk_score", "risk_level",
    "evaluation_finding"
]])

Videos evaluated: 7
Risk report saved: /content/drive/MyDrive/FireWatch-AI/outputs/fire_risk_engine/seven_video_risk_report.csv


,video_name,ground_truth,fire_detection_rate,smoke_detection_rate,risk_score,risk_level,evaluation_finding
0,negsVideo1.858.avi,negative,0.0495,0.0066,3.66,High,Potential false alarm
1,posVideo1.868.avi,fire_positive,1.0000,0.0000,70.00,High,Expected class detected
2,posVideo2.871.avi,fire_positive,1.0000,0.0294,70.88,High,Expected class detected
3,testneg01.807.avi,negative,0.0000,0.0000,0.00,Low,No detections on negative video
4,testneg02.808.avi,negative,0.0000,0.0000,0.00,Low,No detections on negative video
5,testpos01.817.avi,smoke_positive,0.0000,0.5714,17.14,Medium,Expected class detected
6,testpos02.818.avi,smoke_positive,0.3707,0.0000,25.95,High,Potential fire-smoke confusion


In [21]:
import pandas as pd
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/FireWatch-AI")
OUT = PROJECT / "outputs/fire_risk_engine"
risk_path = OUT / "seven_video_risk_report.csv"

df = pd.read_csv(risk_path)

# Consistent score-based levels, with fire-presence safety override
def corrected_level(row):
    if row["fire_detection_rate"] > 0:
        return "Critical" if row["risk_score"] >= 75 else "High"
    if row["risk_score"] >= 50:
        return "High"
    if row["risk_score"] >= 10:
        return "Medium"
    return "Low"

df["risk_level"] = df.apply(corrected_level, axis=1)

# Validate results
assert len(df) == 7
assert df["video_name"].is_unique
assert df["risk_score"].between(0, 100).all()
assert df["risk_level"].isin(["Low", "Medium", "High", "Critical"]).all()
assert (df.loc[df["fire_detection_rate"] > 0, "risk_level"] != "Low").all()

df.to_csv(risk_path, index=False)

validation_path = OUT / "seven_video_validation.txt"
validation_path.write_text(
    "FireWatch AI - Seven Video Evaluation\n"
    "=====================================\n"
    "Videos evaluated: 7\n"
    "Unique video records: PASS\n"
    "Risk score bounds: PASS\n"
    "Risk level values: PASS\n"
    "Fire detection override: PASS\n"
    "Overall: PASS\n\n"
    "Limitations: Small initial sample; risk scores are prototype estimates.\n"
    "Smoke/fire class confusion requires further model evaluation.\n"
)

print("ALL TESTS PASSED")
print("Risk report:", risk_path)
print("Validation:", validation_path)
display(df[["video_name", "risk_score", "risk_level", "evaluation_finding"]])

ALL TESTS PASSED
Risk report: /content/drive/MyDrive/FireWatch-AI/outputs/fire_risk_engine/seven_video_risk_report.csv
Validation: /content/drive/MyDrive/FireWatch-AI/outputs/fire_risk_engine/seven_video_validation.txt


,video_name,risk_score,risk_level,evaluation_finding
0,negsVideo1.858.avi,3.66,High,Potential false alarm
1,posVideo1.868.avi,70.00,High,Expected class detected
2,posVideo2.871.avi,70.88,High,Expected class detected
3,testneg01.807.avi,0.00,Low,No detections on negative video
4,testneg02.808.avi,0.00,Low,No detections on negative video
5,testpos01.817.avi,17.14,Medium,Expected class detected
6,testpos02.818.avi,25.95,High,Potential fire-smoke confusion


In [22]:
import pandas as pd
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/FireWatch-AI")
OUT = PROJECT / "outputs/fire_risk_engine"
risk_path = OUT / "seven_video_risk_report.csv"

df = pd.read_csv(risk_path)

def corrected_level(row):
    # A fire detection triggers High, but flag it separately for review.
    if row["fire_detection_rate"] > 0:
        return "High" if row["ground_truth"] == "negative" else (
            "Critical" if row["risk_score"] >= 75 else "High"
        )

    if row["risk_score"] >= 50:
        return "High"
    if row["risk_score"] >= 10:
        return "Medium"
    return "Low"

df["risk_level"] = df.apply(corrected_level, axis=1)
df.to_csv(risk_path, index=False)

print("Updated risk report")
display(df[[
    "video_name", "ground_truth", "risk_score",
    "risk_level", "evaluation_finding"
]])

Updated risk report


,video_name,ground_truth,risk_score,risk_level,evaluation_finding
0,negsVideo1.858.avi,negative,3.66,High,Potential false alarm
1,posVideo1.868.avi,fire_positive,70.00,High,Expected class detected
2,posVideo2.871.avi,fire_positive,70.88,High,Expected class detected
3,testneg01.807.avi,negative,0.00,Low,No detections on negative video
4,testneg02.808.avi,negative,0.00,Low,No detections on negative video
5,testpos01.817.avi,smoke_positive,17.14,Medium,Expected class detected
6,testpos02.818.avi,smoke_positive,25.95,High,Potential fire-smoke confusion
